# Self-Hostingwith Ollama

**Module 13 · Lesson 13.5**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Why Self-Host: The Crossover
- Ollama In One Command
- Quantization: Fit It On Your VRAM
- Context Costs Memory: The KV Cache
- Wire It In: The OpenAI-Compatible API + Modelfile
- Serve It: Concurrency And The Ceiling

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
!pip install openai -q

# Reproducibility - the lesson uses seeded random values throughout

## No per-token bill, no rate limit, your data stays in - but is it cheaper?

> 💡 **Info**
>
> So far your app has lived on a hosted API. You pay for **every token**, you live inside a provider’s **rate limits**, and your prompts and data **leave your walls** on every request. Self-hosting flips all three: **Ollama** runs an open-weight model — Meta’s Llama, Alibaba’s Qwen, Google’s Gemma — on hardware you control, so there is **no per-token bill**, **no rate limit**, the data **never leaves**, and it runs **offline**. But this is a cost-and-performance module, so the real question is the uncomfortable one: *is it actually cheaper?* Renting an apartment is cheap if you only stay a while; owning a house pays off only if you stay long enough — and self-hosting is exactly that trade. An API is a **variable** per-token cost; a GPU is a **fixed** monthly cost that you pay whether it is busy or idle. Below a **break-even volume** the API wins; above it, the GPU wins — but only if you keep the GPU **busy**, because a machine you barely use costs more per token than any API. This lesson walks the whole path: run a model in one command, fit it on your VRAM (quantization, then the KV cache), wire it into your app (Ollama speaks the OpenAI API), serve it, and finish with the total-cost math that actually decides self-host versus API.

### 🎯 What you will be able to do after this lesson

- **Build** self-hosting tooling — a break-even calculator, a which-model-fits check, a quantization VRAM sizer, a KV-cache context budgeter, a Modelfile parser plus OpenAI-compatible wiring, a serving-knob model, and a real total-cost-of-ownership (TCO) calculator by utilization — as runnable models.

- **Compare** an API’s per-token cost with a GPU’s fixed cost, quantization levels on VRAM and quality, and Ollama with vLLM on concurrency.

- **Operate** `ollama pull` / `run`, a Modelfile, and the `num_ctx`, `OLLAMA_NUM_PARALLEL`, and `KEEP_ALIVE` knobs.

- **Evaluate** a self-host decision: at your volume, quality bar, and *utilization*, does owning the GPU beat renting the API?

> 📦 **Info**
>
> ✅ Before you startIn **13.1** an API was a variable per-token cost; a GPU is a fixed monthly cost, which is why *volume* decides self-host versus API. In **13.3** you routed cheap queries down to a small model — a self-hosted open model is the cheapest possible route target, if you keep it busy. And in **13.4** the KV cache and prefill lived on the provider’s GPU; self-hosting, *you* own that VRAM and the throughput ceiling. Production-scale serving with vLLM comes in **13.6**.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🏠 **Analogy**
>
> Self-hosting versus an API is **owning a house versus renting an apartment**. Renting is easy to start and you pay every month, but the landlord sets the price, can change the terms, and you share the building. Owning costs more up front and you maintain everything yourself — but there is no monthly rent to a landlord, no one else reads your mail, and you can renovate however you like. The decision is pure economics with one twist: a house only beats renting if you *live in it*. A GPU you pay for but barely use is a house you bought and left empty — still cheaper to have rented. **Where it breaks down:** you can walk away from a lease in a month, but a GPU commitment (and the ops work to run it) is stickier, which is why you measure before you buy.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Self-hosting is always cheaper because there is no per-token bill.”** A GPU you underuse costs *more* per token than an API — utilization decides it, not the absence of a per-token line item.
> - **“An open model is a big downgrade.”** Quality is task-specific, open models match or beat frontier on many tasks, and the default quantization costs only about five percent of quality for a four-fold memory saving.

> 💡 **Info**
>
> ⚠️ Anti-patternTwo ways to burn money self-hosting. **Committing to a GPU before measuring your volume and utilization** — the idle-GPU trap, where you pay around the clock for a machine that sits mostly empty and loses to every API on cost per token. And **running Ollama as a high-concurrency production API** — it is one-request-at-a-time by design; serving many concurrent users is vLLM’s job, which is the next lesson.

---

## 🎯 Concept 1: Why Self-Host: The Crossover

### Why Self-Host: The Crossover

An API bills per token (variable); a GPU is a fixed monthly cost (paid whether busy or idle). Below a break-even volume the API is cheaper; above it, the GPU is. Which API you compare to - a pricey frontier one or a cheap open-model one - moves the crossover a lot.

Start with the economics, because they decide everything else. A hosted API is a **variable cost**: you pay per token, so your bill scales with usage — a straight line up from zero. A self-hosted GPU is a **fixed cost**: you pay the same monthly amount whether you push a million tokens through it or none — a flat horizontal line. Those two lines **cross** at a break-even volume: below it the variable API is cheaper, above it the fixed GPU is. The subtlety that trips people up is *which API you compare to*. Against an expensive **frontier API**, the break-even is a moderate monthly volume, so self-hosting wins fairly readily. Against a **cheap hosted open-model API** — the kind that already runs optimized infrastructure at razor-thin margins — the break-even is enormous, so self-hosting rarely wins on cost alone. (Beyond cost, self-hosting also buys privacy, offline operation, and no rate limits — sometimes those alone justify it.) The block computes the crossover, keyless (illustrative prices).

> 🏘️ **Analogy**
>
> It is **renting versus buying a home**, drawn as two lines on a graph. Rent is a line that climbs the longer you stay — every month adds to the total. A mortgage-and-upkeep is a flatter, higher line that barely moves with time. For a short stay, renting is under the buying line (cheaper); past a break-even number of years the lines cross and owning wins. Your token volume is the ‘length of stay’: a little usage and the API (rent) is cheaper; a lot of steady usage and the GPU (own) pulls ahead.

Your self-hosted GPU costs a fixed amount per month. You compare it to two APIs: an expensive frontier one and a cheap open-model one. Against which is the break-even volume HIGHER (harder for self-hosting to win)?

**📝 `01_the_crossover.py`** - *runnable*

In [ ]:
# The crossover: an API bills PER TOKEN (variable), self-hosting is a FIXED monthly cost (the GPU).
# Below the break-even volume the API wins; above it, self-hosting wins. (Illustrative 2026 prices.)
GPU_FIXED = 1095.00          # a rented mid GPU: ~$1.50/hr x 730 hr/month (weights + serving)
API_FRONTIER = 10.00         # $/1M tokens on a frontier API (blended in+out)
API_OPEN = 0.40              # $/1M tokens on a cheap hosted OPEN-model API (Together-class, thin margins)

def breakeven(api_price):    # millions of tokens/month where fixed GPU == API bill
    return GPU_FIXED / api_price
print("Self-host fixed cost: ${:.0f}/month (the GPU is paid whether you use it or not).".format(GPU_FIXED))
print("Break-even volume vs each API (where the GPU cost equals the API bill):")
print("  vs a FRONTIER API (${:.2f}/1M):     {:>6.0f}M tokens/month".format(API_FRONTIER, breakeven(API_FRONTIER)))
print("  vs a cheap OPEN-model API (${:.2f}/1M): {:>6.0f}M tokens/month".format(API_OPEN, breakeven(API_OPEN)))
print()
for vol in [50, 300, 3000]:  # millions of tokens/month
    front, opn, self_host = vol * API_FRONTIER / 1, vol * API_OPEN / 1, GPU_FIXED
    winner = "self-host" if self_host < min(front, opn) else ("open-API" if opn < front else "frontier-API")
    print("  at {:>4}M tok/mo: frontier ${:>6.0f}, open-API ${:>5.0f}, self-host ${:>5.0f}  -> cheapest: {}".format(
        vol, front, opn, self_host, winner))
print()
print("Self-hosting beats a FRONTIER API at moderate volume, but rarely beats a cheap OPEN-model API,")
print("which already runs optimized infra at thin margins. WHICH API you compare to decides the crossover.")

# Output:
# Self-host fixed cost: $1095/month (the GPU is paid whether you use it or not).
# Break-even volume vs each API (where the GPU cost equals the API bill):
#   vs a FRONTIER API ($10.00/1M):        110M tokens/month
#   vs a cheap OPEN-model API ($0.40/1M):   2738M tokens/month
#
#   at   50M tok/mo: frontier $   500, open-API $   20, self-host $ 1095  -> cheapest: open-API
#   at  300M tok/mo: frontier $  3000, open-API $  120, self-host $ 1095  -> cheapest: open-API
#   at 3000M tok/mo: frontier $ 30000, open-API $ 1200, self-host $ 1095  -> cheapest: self-host
#
# Self-hosting beats a FRONTIER API at moderate volume, but rarely beats a cheap OPEN-model API,
# which already runs optimized infra at thin margins. WHICH API you compare to decides the crossover.

- The GPU is a **fixed monthly cost** paid whether or not you use it; the API bill scales with volume.
- The **break-even** is the GPU cost divided by the API price — a moderate volume against a frontier API, an enormous one against a cheap open-model API.
- At low volume the **API wins**; only at high volume does the fixed GPU pull ahead, and mostly versus the frontier API.
- The lesson: the crossover is real, but *which* API you compare to moves it by orders of magnitude — compare to the right alternative.

#### 💡 What just happened

⚡ What just happened? An API is a variable per-token cost, a GPU is a fixed monthly cost, and they cross at a break-even volume - below it the API wins, above it the GPU, and which API you compare to moves the crossover enormously. Tradeoff / the framing: self-hosting trades a variable bill for a fixed one plus the work of running it. Next: how you actually run a model - in one command.

- A volume slider; a flat self-host cost line and two sloped API lines (frontier, open) cross it
- Below break-even the API wins; above it self-host wins - and the cheap open-API line rarely gets beaten

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Ollama In One Command

### Ollama In One Command

Ollama is llama.cpp plus a daemon, a CLI, and a model registry. `ollama pull` then `ollama run` downloads an open model and gives you an OpenAI-compatible endpoint on localhost:11434 in about thirty seconds. The one hard rule: the model must fit in memory, or it swaps to disk and crawls.

Running a model locally used to be a weekend project; Ollama makes it one command. **Ollama** is the popular open-source runtime built on **llama.cpp**, wrapped in a background daemon, a command-line tool, an HTTP API, and a model registry. You `ollama pull` a model — it downloads like a Docker image — and `ollama run` starts an interactive chat *and* a local server on **localhost:11434** with an **OpenAI-compatible** endpoint, all in about thirty seconds, with no API key and no signup. The registry carries open weights across families: Meta’s **Llama**, Alibaba’s **Qwen** (whose coder models rival frontier models on code), Google’s **Gemma**, plus Mistral, DeepSeek, and Microsoft’s Phi. These are open *weights*, but the **licenses vary** — some are fully permissive (Apache-2.0), others carry use restrictions or scale thresholds — so check a model’s license before you ship it commercially. The one rule you cannot break: the **model must fit in memory** — VRAM on a GPU, or unified memory on a Mac. If it does not fit, it spills to disk and becomes drastically slower, which is why the next two steps are about making a model fit. The block checks which models fit a given GPU, keyless.

> 📱 **Analogy**
>
> Ollama is **an app store for language models**. You do not compile anything or hunt for weights on a dozen sites; you browse a registry, tap install (`ollama pull`), and the model is on your machine ready to run — exactly like installing an app on a phone. And just like a phone app, it has a system requirement printed on the box: it needs enough memory to load, or it will not run smoothly. The registry is the store; `pull` is install; `run` is open; and the memory requirement is the ‘this app needs iOS 18’ line you have to check first.

You have a GPU with 24 GB of VRAM. You try to `ollama run` a 70B model that needs about 45 GB at its usual quantization. What happens?

**📝 `02_which_models_fit.py`** - *runnable*

In [ ]:
# Ollama = llama.cpp + a daemon + a CLI + an HTTP API + a model registry. One command pulls a model
# and gives you an OpenAI-compatible endpoint on localhost:11434. Which 2026 open models fit YOUR GPU?
GPU_VRAM = 24                # your GPU, in GB (illustrative)
catalog = [                  # (ollama tag, params_B, note) - Q4_K_M VRAM approx = params * 4.5/8 + ~15% overhead
    ("llama3.2:3b",          3,  "tiny, fast, on a laptop"),
    ("qwen2.5-coder:7b",     7,  "strong coding for its size"),
    ("phi4:14b",            14,  "small-but-capable reasoning"),
    ("gemma3:27b",          27,  "high quality, mid GPU"),
    ("qwen2.5-coder:32b",   32,  "~92% HumanEval, code specialist"),
    ("llama3.3:70b",        70,  "frontier-class open weights"),
]
def q4_vram(p): return round(p * 4.5 / 8 * 1.15, 1)   # Q4_K_M bytes-per-param ~4.5 bits + runtime overhead
print("Ollama catalog at Q4_K_M vs a {} GB GPU (pull with: ollama pull <tag>):".format(GPU_VRAM))
for tag, p, note in catalog:
    v = q4_vram(p)
    fits = "FITS" if v <= GPU_VRAM else "too big"
    print("  {:<20} {:>2}B  ~{:>4.1f} GB  [{:<7}]  {}".format(tag, p, v, fits, note))
print()
biggest = max((p for tag, p, note in catalog if q4_vram(p) <= GPU_VRAM), default=0)
print("Biggest model that fits: {}B at Q4_K_M. `ollama run <tag>` and you have a local OpenAI endpoint in ~30s.".format(biggest))
print("The registry has Llama, Qwen, Gemma, Mistral, DeepSeek, Phi - all open weights, all one pull away.")

# Output:
# Ollama catalog at Q4_K_M vs a 24 GB GPU (pull with: ollama pull <tag>):
#   llama3.2:3b           3B  ~ 1.9 GB  [FITS   ]  tiny, fast, on a laptop
#   qwen2.5-coder:7b      7B  ~ 4.5 GB  [FITS   ]  strong coding for its size
#   phi4:14b             14B  ~ 9.1 GB  [FITS   ]  small-but-capable reasoning
#   gemma3:27b           27B  ~17.5 GB  [FITS   ]  high quality, mid GPU
#   qwen2.5-coder:32b    32B  ~20.7 GB  [FITS   ]  ~92% HumanEval, code specialist
#   llama3.3:70b         70B  ~45.3 GB  [too big]  frontier-class open weights
#
# Biggest model that fits: 32B at Q4_K_M. `ollama run <tag>` and you have a local OpenAI endpoint in ~30s.
# The registry has Llama, Qwen, Gemma, Mistral, DeepSeek, Phi - all open weights, all one pull away.

- Each catalog model is sized at Q4_K_M and checked against the GPU’s VRAM — the ones that **fit** light up, the ones too big do not.
- The **biggest model that fits** is the one to pull; `ollama run` then gives a local OpenAI-compatible endpoint in about thirty seconds.
- The registry spans **Llama, Qwen, Gemma, Mistral, DeepSeek, and Phi** — all open weights, all one `pull` away.
- The hard rule: a model must fit in memory, so the very next question is how to make a big model small enough — quantization.

#### 💡 What just happened

⚡ What just happened? Ollama (llama.cpp + a daemon + a CLI + a registry) pulls an open model and serves it on localhost:11434 with an OpenAI-compatible endpoint in ~30 seconds - as long as the model fits in memory. Tradeoff: you trade a hosted endpoint for a local one you have to size to your hardware. Next: quantization, which is how you make a big model fit a small GPU.

- A GPU-VRAM slider; the open-model catalog lights green for models that fit at Q4 and greys the too-big ones
- The biggest model that fits is called out - `ollama run` and you have a local OpenAI endpoint

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Quantization: Fit It On Your VRAM

### Quantization: Fit It On Your VRAM

Quantization stores each weight in fewer bits so a big model fits a small GPU. VRAM is roughly parameters times bits over eight. Q4_K_M is the 2026 sweet spot: about 95 percent of full-precision quality at roughly four times smaller. Below Q4, reasoning and code degrade first.

A model’s size is set by two things: how many **parameters** it has, and how many **bits** each parameter is stored in. Full precision (FP16) is sixteen bits per weight, which is why an eight-billion-parameter model needs roughly sixteen gigabytes of VRAM — the sizing rule is simply **parameters times bits, over eight**. **Quantization** shrinks the model by storing each weight in fewer bits: eight bits (Q8), or about four and a half (**Q4_K_M**), or fewer. The magic is how *little* quality this costs. **Q4_K_M is the 2026 default sweet spot**: it keeps about **ninety-five percent** of full-precision quality while cutting memory roughly **four-fold**, so that same eight-billion model drops from a sixteen-gigabyte requirement to about five — the difference between needing a data-center card and running on a laptop GPU. Push below Q4 (Q3, Q2) only when you must, because degradation becomes noticeable and **reasoning and code tasks degrade first and most**. The format Ollama ships is GGUF, and Q4_K_M is what most people should pull. The block sizes the levels, keyless.

> 📷 **Analogy**
>
> Quantization is **saving a photo as a JPEG**. The original raw image is enormous and pixel-perfect (that is FP16). A high-quality JPEG throws away detail your eye cannot see and lands at a fraction of the size, looking essentially identical (that is Q4_K_M — about ninety-five percent of the quality at a quarter of the size). Crank the JPEG compression too far and you start seeing blocky artifacts (that is Q2 — visible degradation, worst on the fine detail, which for a model is reasoning and code). You pick the compression that fits your storage while still looking right; you pick the quantization that fits your VRAM while still answering well.

An 8B model needs about 16 GB at full FP16 precision. You have an 8 GB laptop GPU. Which quantization lets it run, and at what quality cost?

**📝 `03_quantization.py`** - *runnable*

In [ ]:
# Quantization shrinks the weights so a big model fits a small GPU. VRAM ~= params x bits / 8.
# Q4_K_M is the 2026 sweet spot: ~95% of full-precision quality at ~4x smaller. Below Q4, quality drops.
params_B = 8                 # an 8B model
levels = [("FP16 (full)", 16, 1.00), ("Q8_0", 8, 0.99), ("Q4_K_M", 4.5, 0.95), ("Q2_K", 2.6, 0.82)]
print("An {}B model at different quantization levels (VRAM ~= params x bits / 8, + ~15% overhead):".format(params_B))
for name, bits, quality in levels:
    vram = round(params_B * bits / 8 * 1.15, 1)
    print("  {:<12} {:>4} bits  ~{:>4.1f} GB  quality ~{:.0%}".format(name, bits, vram, quality))
print()
fp16 = params_B * 16 / 8 * 1.15
q4 = params_B * 4.5 / 8 * 1.15
print("Q4_K_M cuts VRAM {:.1f}x ({:.1f} GB -> {:.1f} GB) for only about a 5% quality drop - the default choice.".format(fp16 / q4, fp16, q4))
print("An 8B model in FP16 needs a 24 GB card; at Q4_K_M (~5 GB) it runs on an 8 GB laptop GPU.")
print("Go below Q4 (Q2, Q3) only when you must - reasoning and code degrade first and most.")

# Output:
# An 8B model at different quantization levels (VRAM ~= params x bits / 8, + ~15% overhead):
#   FP16 (full)    16 bits  ~18.4 GB  quality ~100%
#   Q8_0            8 bits  ~ 9.2 GB  quality ~99%
#   Q4_K_M        4.5 bits  ~ 5.2 GB  quality ~95%
#   Q2_K          2.6 bits  ~ 3.0 GB  quality ~82%
#
# Q4_K_M cuts VRAM 3.6x (18.4 GB -> 5.2 GB) for only about a 5% quality drop - the default choice.
# An 8B model in FP16 needs a 24 GB card; at Q4_K_M (~5 GB) it runs on an 8 GB laptop GPU.
# Go below Q4 (Q2, Q3) only when you must - reasoning and code degrade first and most.

- VRAM is **parameters times bits over eight** (plus overhead), so fewer bits per weight means a smaller model.
- **Q4_K_M** cuts VRAM roughly four-fold versus FP16 for only about a five percent quality drop — the default choice.
- That is what lets a model that needed a big card run on a **laptop GPU** instead.
- Go below Q4 only when you must — **reasoning and code degrade first and most**, so aggressive quantization hurts exactly the hard tasks.

#### 💡 What just happened

⚡ What just happened? Quantization stores weights in fewer bits so a big model fits a small GPU; VRAM ~= params x bits / 8, and Q4_K_M keeps ~95 percent quality at ~4x smaller - the default. Tradeoff: a small, mostly-invisible quality loss in exchange for fitting a far bigger model on your hardware. Next: the OTHER thing that eats VRAM - the context window.

- A quant-level slider (FP16 to Q2) shrinking a model’s VRAM bar
- A quality gauge dips slowly then falls off a cliff below Q4 - Q4_K_M is the sweet spot

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Context Costs Memory: The KV Cache

### Context Costs Memory: The KV Cache

Total VRAM is not just the weights - it is weights plus the KV cache, and the KV cache grows linearly with the context length. So a long context you set with num_ctx eats real gigabytes. Budget num_ctx to what you need, or it steals VRAM you could spend on a bigger model.

Here is the memory cost that catches people by surprise: the **context window is not free**. When a model processes your prompt it stores an attention state — the **KV cache** — for every token, and it keeps that cache in VRAM alongside the weights. So your real memory budget is **weights plus KV cache**, and the KV cache **grows linearly with the context length**: double the context, double that cache. For a mid-size model the KV cache runs on the order of a hundred kilobytes per token, so a thirty-two-thousand-token context adds several gigabytes, and a hundred-and-twenty-eight-thousand-token context can add *more than the weights themselves*. This is why `num_ctx` (Ollama’s context-length setting) is a real budget knob, not a free dial to crank up: a giant context you never actually fill just burns VRAM you could have spent on a bigger, smarter model or on serving more requests at once. Set `num_ctx` to what your task needs, and no larger. The block budgets the context, keyless.

> 📑 **Analogy**
>
> The KV cache is **a desk that has to hold every note the model has taken so far**. The model (the weights) is the worker, and it always takes up the same amount of room. But for a conversation it keeps a running set of notes — one sticky note per token — spread out on the desk so it can glance back at any of them. A short chat is a few notes; a book-length context is the desk buried under thousands of sheets. A *bigger notebook needs a bigger desk*: the longer the context, the more desk space (VRAM) the notes consume, on top of the fixed space the worker already occupies.

Your 8B model at Q4 uses about 5 GB of weights and fits your 12 GB GPU fine at a short context. You bump num_ctx up to 128K. What is most likely?

**📝 `04_kv_cache_budget.py`** - *runnable*

In [ ]:
# The context window costs VRAM too. Total = model weights + the KV cache, and the KV cache grows
# LINEARLY with context length. So your real memory budget is weights + KV, not just the weights.
# KV bytes/token = 2 (K and V) x n_layers x n_kv_heads x head_dim x 2 bytes (fp16). (Llama-3-8B GQA numbers.)
n_layers, n_kv_heads, head_dim = 32, 8, 128
weights_gb = 4.7            # 8B at Q4_K_M on disk (~5 GB, the Step 3 estimate)
kv_per_token = 2 * n_layers * n_kv_heads * head_dim * 2      # bytes per token
def kv_gb(ctx): return round(kv_per_token * ctx / 1e9, 2)
print("KV cache per token: {:,} bytes ({:.0f} KB). It is paid for EVERY token in the context window.".format(kv_per_token, kv_per_token / 1024))
print("Total VRAM = {} GB weights + KV cache:".format(weights_gb))
for ctx in [8192, 32768, 131072]:
    total = round(weights_gb + kv_gb(ctx), 1)
    print("  num_ctx {:>6}:  weights {} GB + KV {:>4.1f} GB = {:>4.1f} GB total".format(ctx, weights_gb, kv_gb(ctx), total))
print()
budget = 12                # a 12 GB GPU
max_kv = budget - weights_gb - 0.8    # leave headroom for activations
max_ctx = int(max_kv * 1e9 / kv_per_token)
print("On a {} GB GPU, the KV cache can use ~{:.1f} GB, so the max context is about {:,} tokens.".format(budget, max_kv, max_ctx // 1000 * 1000))
print("Set num_ctx no larger than you need: a long context you never use just burns VRAM you could spend on a bigger model.")

# Output:
# KV cache per token: 131,072 bytes (128 KB). It is paid for EVERY token in the context window.
# Total VRAM = 4.7 GB weights + KV cache:
#   num_ctx   8192:  weights 4.7 GB + KV  1.1 GB =  5.8 GB total
#   num_ctx  32768:  weights 4.7 GB + KV  4.3 GB =  9.0 GB total
#   num_ctx 131072:  weights 4.7 GB + KV 17.2 GB = 21.9 GB total
#
# On a 12 GB GPU, the KV cache can use ~6.5 GB, so the max context is about 49,000 tokens.
# Set num_ctx no larger than you need: a long context you never use just burns VRAM you could spend on a bigger model.

- The **KV cache** costs a fixed number of bytes per token — and you pay it for every token in the context window.
- Total VRAM is **weights plus KV cache**, and the KV cache grows with `num_ctx` — a huge context can cost more than the weights.
- On a given GPU, the weights fix a floor and the **remaining VRAM caps the context** you can run.
- The lesson: set `num_ctx` to what you need — an unused long context just steals VRAM from a bigger model or more concurrency.

#### 💡 What just happened

⚡ What just happened? Total VRAM is weights + the KV cache, and the KV cache grows linearly with the context length, so num_ctx is a real memory budget - a huge context can cost more than the weights. Tradeoff: a longer context buys the model more to work with, at a VRAM cost you could instead spend on a bigger model or more concurrency. Next: wiring the model into your app.

- A num_ctx slider grows a KV-cache bar stacked on a fixed weights bar
- The total creeps toward a GPU-capacity line and turns rose when it overflows

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Wire It In: The OpenAI-Compatible API + Modelfile

### Wire It In: The OpenAI-Compatible API + Modelfile

Ollama speaks the OpenAI API, so moving an app to a local model is a one-line change: point your OpenAI client at localhost:11434/v1. A Modelfile (FROM, SYSTEM, PARAMETER) bakes a system prompt and params into a named, reusable model you build with ollama create.

The best part for an engineer: you barely have to change your code. Ollama exposes an **OpenAI-compatible API**, which means your existing OpenAI client works against it unchanged — you point the `base_url` at **localhost:11434/v1** (with any placeholder api_key), and everything downstream is the *same call*: streaming, tool use, JSON mode, temperature, all identical. Porting an app from a hosted API to a self-hosted model is usually an **afternoon, not a rewrite**. The second piece is the **Modelfile**: a small text file with a few directives — `FROM` (the base model), `SYSTEM` (a baked-in system prompt), and `PARAMETER` (settings like `temperature` and `num_ctx`) — that you build into a named, reusable model with `ollama create`. It is the local equivalent of a saved assistant configuration: bake the persona and params once, then just call the model by name. The block parses a Modelfile and shows the one-line wiring, keyless; the illustrative block after it makes the live call.

> 🔌 **Analogy**
>
> The OpenAI-compatible API is **a universal power adapter**. Your app is an appliance with a plug shaped like the OpenAI protocol. A hosted API is one wall socket; your local Ollama server is another socket in a different room — but the plug is the same, so all you do is move to the other outlet (change the `base_url`). You do not rewire the appliance. And the Modelfile is like presetting a device: you dial in the settings once, save it under a name, and from then on you just switch it on — no reconfiguring each time.

Your app calls a hosted API through the OpenAI Python client. You want it to use a local Ollama model instead. How much changes?

**📝 `05_wire_it_in.py`** - *runnable*

In [ ]:
# Ollama speaks the OpenAI API, so wiring an app in is a ONE-LINE base_url change. A Modelfile bakes a
# system prompt + params into a named, reusable model. Here we PARSE a Modelfile and show the request
# SHAPE deterministically (no live server needed); the live call is the illustrative block below.
MODELFILE = """FROM qwen2.5-coder:32b
SYSTEM You are a terse senior code reviewer. Reply with a diff, no prose.
PARAMETER temperature 0.2
PARAMETER num_ctx 16384"""
directives = {}
for line in MODELFILE.strip().splitlines():
    key, _, val = line.partition(" ")
    directives.setdefault(key, []).append(val)
print("A Modelfile bakes config into a reusable model (build it: ollama create my-reviewer -f Modelfile):")
for key in ["FROM", "SYSTEM", "PARAMETER"]:
    print("  {:<10} {}".format(key, "  |  ".join(directives.get(key, []))))
print()
CLOUD = "https://api.openai.com/v1"
LOCAL = "http://localhost:11434/v1"
print("Wiring an existing app to the local model is a ONE-LINE change:")
print("  cloud:  base_url = {}".format(CLOUD))
print("  local:  base_url = {}   (api_key is just a placeholder)".format(LOCAL))
print("Everything downstream - streaming, tools, JSON mode - is the SAME OpenAI-compatible call.")
print("So porting an app from a hosted API to a self-hosted model is usually an afternoon, not a rewrite.")

# Output:
# A Modelfile bakes config into a reusable model (build it: ollama create my-reviewer -f Modelfile):
#   FROM       qwen2.5-coder:32b
#   SYSTEM     You are a terse senior code reviewer. Reply with a diff, no prose.
#   PARAMETER  temperature 0.2  |  num_ctx 16384
#
# Wiring an existing app to the local model is a ONE-LINE change:
#   cloud:  base_url = https://api.openai.com/v1
#   local:  base_url = http://localhost:11434/v1   (api_key is just a placeholder)
# Everything downstream - streaming, tools, JSON mode - is the SAME OpenAI-compatible call.
# So porting an app from a hosted API to a self-hosted model is usually an afternoon, not a rewrite.

**📝 `self_hosted_client.py`** - *illustrative*

In [ ]:
# SELF-HOSTED CLIENT - an illustrative minimal example (the OpenAI client, pointed at a local Ollama).
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")   # the ONLY change vs a hosted API

def review(code: str):
    # streaming works exactly as it would against a hosted API - Ollama speaks the OpenAI protocol
    stream = client.chat.completions.create(
        model="my-reviewer",                 # a Modelfile-built model: ollama create my-reviewer -f Modelfile
        messages=[{"role": "user", "content": code}],
        stream=True,
    )
    for chunk in stream:
        yield chunk.choices[0].delta.content or ""     # the same token stream your UI already handles
# Everything else - tools, JSON mode, temperature - is the identical OpenAI call, now running on YOUR GPU.
# Output: (illustrative minimal example - needs a running Ollama + the model pulled; not run here.)

- A **Modelfile** (`FROM` + `SYSTEM` + `PARAMETER`) bakes a persona and params into a named model you build with `ollama create`.
- Wiring an app to the local model is a **one-line change** — point the OpenAI client’s `base_url` at `localhost:11434/v1`.
- Everything downstream — **streaming, tools, JSON mode** — is the same OpenAI-compatible call, now running on your own GPU.
- The **illustrative client** shows the live version: the standard OpenAI client streaming from a Modelfile-built local model.

#### 💡 What just happened

⚡ What just happened? Ollama speaks the OpenAI API, so wiring an app to a local model is a one-line base_url change to localhost:11434/v1; a Modelfile (FROM/SYSTEM/PARAMETER) bakes config into a named model via ollama create. Tradeoff: none on the code side - the protocol is identical; the work is running the server. Next: serving it under real load.

- A base_url toggle swaps a request from the cloud endpoint to localhost:11434, the rest of the call unchanged
- A Modelfile (FROM / SYSTEM / PARAMETER) builds into a named model

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Serve It: Concurrency And The Ceiling

### Serve It: Concurrency And The Ceiling

Two knobs matter: OLLAMA_NUM_PARALLEL (concurrent requests per model) and OLLAMA_KEEP_ALIVE (how long the model stays in VRAM). Queue depth past twice NUM_PARALLEL doubles p95. But Ollama serves one-at-a-time at its core; for many concurrent users you graduate to vLLM - roughly 17x the throughput (Lesson 13.6).

Running one model for yourself is easy; serving it to real traffic needs two settings and one honest limit. **`OLLAMA_NUM_PARALLEL`** sets how many requests a model handles at once (a small default); the rest **queue**. **`OLLAMA_KEEP_ALIVE`** keeps the model loaded in VRAM so it does not have to reload — set it too short and every burst of traffic pays a multi-second reload penalty. The metric to watch is **queue depth**: once it passes roughly twice your `NUM_PARALLEL`, p95 latency doubles, which is your signal to add replicas behind a load balancer. But here is the ceiling you have to respect: Ollama’s llama.cpp foundation is built for **one user at a time**, so it does not truly batch concurrent requests. For **many concurrent users**, **vLLM** — with continuous batching and PagedAttention — delivers *roughly seventeen times* the throughput on the same GPU. Ollama is the right tool for development and low concurrency; when you outgrow it you graduate to vLLM, which is the next lesson. One safety note before you serve real traffic: Ollama binds to localhost with **no built-in authentication** (that placeholder api_key is exactly that — no key at all), so the moment you expose it beyond localhost you must put an authenticated gateway in front — unauthenticated Ollama servers get found and abused on the public internet. The block models the serving knobs, keyless.

> 🍔 **Analogy**
>
> Ollama is **a food truck**; vLLM is **a restaurant kitchen**. A food truck is perfect for a small line: one cook, a few orders at a time, fast to set up, great for you and a handful of customers. But when a stadium crowd shows up, the single cook is overwhelmed — the line stretches around the block. A restaurant kitchen has many stations working in parallel and batches orders efficiently, so it feeds hundreds at once. You do not knock down the food truck to build a kitchen until the crowd demands it — and that is exactly the Ollama-to-vLLM graduation: right tool, right scale.

Your Ollama server is snappy in development, but under production traffic with many simultaneous users, latency spikes badly. What is the real fix?

**📝 `06_serving.py`** - *runnable*

In [ ]:
# Serving in production: Ollama loads ONE model and serves a few requests at once. Two knobs matter.
# OLLAMA_NUM_PARALLEL = concurrent requests per model; OLLAMA_KEEP_ALIVE = how long the model stays in VRAM.
NUM_PARALLEL = 4            # concurrent slots (default 4, or 1 if VRAM is tight)
KEEP_ALIVE_MIN = 5         # keep the model loaded 5 min so a burst does not trigger a 4-6s reload
reload_penalty_s = 5
print("Ollama serving knobs:")
print("  OLLAMA_NUM_PARALLEL={}: up to {} requests run at once; the rest queue.".format(NUM_PARALLEL, NUM_PARALLEL))
print("  OLLAMA_KEEP_ALIVE={}m: model stays in VRAM; too low (30s) and every burst reloads (+{}s latency).".format(KEEP_ALIVE_MIN, reload_penalty_s))
print()
# Queue depth is the cardinal metric: past 2 x NUM_PARALLEL, p95 latency doubles.
for queue in [NUM_PARALLEL, 2 * NUM_PARALLEL, 4 * NUM_PARALLEL]:
    state = "healthy" if queue <= 2 * NUM_PARALLEL else "p95 doubling - add replicas / a load balancer"
    print("  queue depth {:>2}:  {}".format(queue, state))
print()
# The ceiling: Ollama is one-user-at-a-time (llama.cpp); vLLM's continuous batching serves many at once.
ollama_tps, vllm_tps = 484, 8033      # illustrative Llama-70B on a Blackwell GPU
print("Ollama vs vLLM on the same GPU (illustrative Llama-70B): {} vs {:,} tokens/s = {:.0f}x throughput.".format(ollama_tps, vllm_tps, vllm_tps / ollama_tps))
print("Ollama is the right tool for dev and low concurrency; for many concurrent users you graduate to vLLM (Lesson 13.6).")

# Output:
# Ollama serving knobs:
#   OLLAMA_NUM_PARALLEL=4: up to 4 requests run at once; the rest queue.
#   OLLAMA_KEEP_ALIVE=5m: model stays in VRAM; too low (30s) and every burst reloads (+5s latency).
#
#   queue depth  4:  healthy
#   queue depth  8:  healthy
#   queue depth 16:  p95 doubling - add replicas / a load balancer
#
# Ollama vs vLLM on the same GPU (illustrative Llama-70B): 484 vs 8,033 tokens/s = 17x throughput.
# Ollama is the right tool for dev and low concurrency; for many concurrent users you graduate to vLLM (Lesson 13.6).

- `OLLAMA_NUM_PARALLEL` sets concurrent slots; `OLLAMA_KEEP_ALIVE` keeps the model resident so a burst does not trigger a slow reload.
- **Queue depth** is the cardinal metric — past about twice `NUM_PARALLEL`, p95 doubles, so you add replicas.
- Ollama is **one-request-at-a-time** at its core; vLLM’s batching gives roughly seventeen times the throughput on the same GPU.
- So Ollama is right for dev and low concurrency, and you graduate to **vLLM** for many concurrent users (Lesson 13.6).

#### 💡 What just happened

⚡ What just happened? Serving Ollama turns on two knobs (NUM_PARALLEL, KEEP_ALIVE) and watches queue depth (past 2x, p95 doubles); but Ollama is one-at-a-time, so for concurrent load you graduate to vLLM (~17x throughput). Tradeoff: Ollama’s simplicity in exchange for a concurrency ceiling you outgrow at scale. Next: the honest total-cost decision.

- A request-rate slider fills the NUM_PARALLEL slots then a queue; past 2x the slots, p95 doubles (rose)
- A toggle swaps Ollama’s one-at-a-time lane for vLLM’s batched lanes

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Real Decision: TCO And Utilization

### The Real Decision: TCO And Utilization

The naive crossover misses two costs: ops labor (10-20 hours a month, so raw GPU is often only a third of the true bill) and utilization (an idle GPU is expensive per token). Effective cost is the total divided by tokens you actually serve. Self-host at high steady volume; stay on the API when volume is low or spiky.

Now the honest math: the real **total cost of ownership (TCO)** is much more than the GPU’s sticker price, because the naive break-even from Step 1 leaves out two costs that flip real decisions. First, **ops labor**: a self-hosted deployment needs roughly ten to twenty hours a month of engineer time — patching, monitoring, on-call — so the **raw GPU cost is often only a third to a half of the true monthly bill**. Second, and bigger, **utilization**: your effective cost per token is the total monthly cost divided by the tokens you *actually serve*, not the GPU’s theoretical capacity. A GPU running at low utilization is **expensive per token**, because you pay for it around the clock whether it is busy or idle — the idle-GPU trap where, at a few percent utilization, you lose to *every* API on cost. Near saturation, the same GPU can crush a frontier API on cost per token — but even then it usually still loses to a cheap open-model API. So the honest decision: **self-host when volume is high and steady** (utilization high), you need **privacy or offline**, or a **good-enough open model** clears your bar; **stay on the API when volume is low or spiky** (utilization low), you need **frontier quality**, or you have **no ops capacity**. Most teams land on a hybrid. The block runs the real TCO, keyless.

> 🚗 **Analogy**
>
> Owning a GPU is **owning a car**. The sticker price is only the start: there is insurance, maintenance, and the loan every month whether you drive or not (that is ops labor plus the always-on GPU). And the cost that really decides it is how much you *drive* — a car you use daily for a long commute is cheap per mile, but a car you drive twice a month is absurdly expensive per mile, and you would have been better off with taxis (the API). The GPU is that car: cheap per token only if you keep it busy; a money pit if it sits in the driveway.

Your naive break-even (GPU sticker price vs API bill) says self-host. But the GPU will run at only about five percent utilization. What does the real cost look like?

**📝 `07_tco_utilization.py`** - *runnable*

In [ ]:
# The REAL decision is TCO at YOUR utilization - not the sticker GPU price. Two costs the naive crossover
# misses: OPS labor, and low UTILIZATION (a GPU you pay for 24/7 but barely use is expensive per token).
GPU_FIXED = 1095.00                 # rented GPU / month
OPS_HOURS, OPS_RATE = 15, 100       # ~15 hrs/month of engineer time at $100/hr (patching, monitoring, on-call)
ops_cost = OPS_HOURS * OPS_RATE
total_monthly = GPU_FIXED + ops_cost
CAPACITY_M = 6500                   # tokens/month the GPU could serve if kept BUSY (illustrative, millions)
print("True monthly TCO: ${:.0f} GPU + ${:.0f} ops ({} hrs x ${}) = ${:.0f}/month.".format(GPU_FIXED, ops_cost, OPS_HOURS, OPS_RATE, total_monthly))
print("Raw GPU is only about {:.0%} of the real cost - ops labor is the rest, and it never sleeps.".format(GPU_FIXED / total_monthly))
print()
API_FRONTIER, API_OPEN = 10.00, 0.40   # $/1M on a frontier API and on a cheap open-model API (from Step 1)
print("Effective cost per 1M tokens depends entirely on UTILIZATION (actual volume vs the {:,}M capacity):".format(CAPACITY_M))
for util in [0.03, 0.20, 0.80]:
    vol = CAPACITY_M * util
    per_m = total_monthly / vol
    vs_front = "beats" if per_m < API_FRONTIER else "LOSES to"
    vs_open = "beats" if per_m < API_OPEN else "LOSES to"
    print("  util {:>3.0%} ({:>5.0f}M tok/mo):  ${:>6.2f}/1M  ->  {} the ${:.0f} frontier API, {} the ${:.2f} open-API".format(
        util, vol, per_m, vs_front, API_FRONTIER, vs_open, API_OPEN))
print()
print("At tiny utilization the idle GPU loses to EVERYTHING; near saturation it crushes a frontier API but still")
print("loses to a cheap open-model API. Self-hosting is a bet on keeping the GPU BUSY.")
print("Self-host when volume is HIGH and STEADY, you need privacy/offline, or the open model is good enough.")
print("Stay on the API when volume is low or spiky (low util), you need frontier quality, or you have no ops capacity.")

# Output:
# True monthly TCO: $1095 GPU + $1500 ops (15 hrs x $100) = $2595/month.
# Raw GPU is only about 42% of the real cost - ops labor is the rest, and it never sleeps.
#
# Effective cost per 1M tokens depends entirely on UTILIZATION (actual volume vs the 6,500M capacity):
#   util  3% (  195M tok/mo):  $ 13.31/1M  ->  LOSES to the $10 frontier API, LOSES to the $0.40 open-API
#   util 20% ( 1300M tok/mo):  $  2.00/1M  ->  beats the $10 frontier API, LOSES to the $0.40 open-API
#   util 80% ( 5200M tok/mo):  $  0.50/1M  ->  beats the $10 frontier API, LOSES to the $0.40 open-API
#
# At tiny utilization the idle GPU loses to EVERYTHING; near saturation it crushes a frontier API but still
# loses to a cheap open-model API. Self-hosting is a bet on keeping the GPU BUSY.
# Self-host when volume is HIGH and STEADY, you need privacy/offline, or the open model is good enough.
# Stay on the API when volume is low or spiky (low util), you need frontier quality, or you have no ops capacity.

- True TCO is the GPU plus **ops labor** — and ops is often the larger share, so the raw GPU price undersells the real cost.
- Effective cost per token is the total divided by the tokens you **actually serve**, so it depends entirely on **utilization**.
- At tiny utilization the idle GPU **loses to every API**; near saturation it beats a frontier API but still usually loses to a cheap open-model API.
- The decision: self-host at **high, steady utilization** (or for privacy/offline); stay on the API when volume is low or spiky or you lack ops capacity.

#### 💡 What just happened

⚡ What just happened? The real decision is TCO at your utilization: raw GPU is often only a third of the bill (ops labor is the rest), and effective cost = total / tokens actually served, so an idle GPU loses to every API. Tradeoff: self-hosting is a bet on keeping the GPU busy. That is the whole lesson: own the GPU only when volume is high and steady, or privacy demands it - otherwise rent the API.

- A utilization slider; the effective cost-per-1M curve drops as the GPU fills
- It crosses the frontier-API line, then flattens above the cheap open-API floor it never reaches

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Self-hosting is an economics decision wrapped around a few hardware facts. It starts from the **crossover**: an API is a variable per-token cost, a GPU a fixed monthly one, and they cross at a break-even volume that depends on which API you compare to (Step 1). You **run a model in one command** with Ollama, as long as it fits in memory (Step 2), and you make it fit with **quantization** — Q4_K_M keeps most of the quality at a quarter of the VRAM (Step 3) — while budgeting the **KV cache**, because the context window costs real memory too (Step 4). You **wire it in** with a one-line base_url change and a Modelfile (Step 5), **serve it** with the two knobs while respecting Ollama’s one-at-a-time ceiling (Step 6), and finish with the **real TCO**, where ops labor and utilization decide whether owning beats renting (Step 7). Ask two questions of any self-host decision: does your volume clear the **break-even**, and can you keep the GPU **busy**?

| Factor | Lean API | Lean self-host |
|---|---|---|
| Monthly volume | low or spiky | high and steady |
| Utilization | the GPU would sit idle | the GPU stays busy |
| Quality bar | needs frontier quality | a good-enough open model clears it |
| Data privacy | not critical | must never leave your walls |
| Offline | always online | must run without internet |
| Ops capacity | none to spare | can run a GPU (and vLLM at scale) |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now run an open model locally and do the math on whether to. The moment you need to serve **many concurrent users**, Ollama’s one-at-a-time serving is the bottleneck — you graduate to production-scale serving with vLLM and GKE, which comes in **Lesson 13.6**. Self-hosting as an LLMOps serving discipline is **Lesson 14.5**, packaging the server is the Docker of **12.5**, and autoscaling it is the scaling of **12.7**. The primary references are the [Ollama](https://github.com/ollama/ollama) project and its [OpenAI-compatible API](https://github.com/ollama/ollama/blob/main/docs/openai.md), [llama.cpp](https://github.com/ggml-org/llama.cpp) for GGUF quantization, and the [Ollama vs vLLM](https://developers.redhat.com/articles/2025/08/08/ollama-vs-vllm-deep-dive-performance-benchmarking) benchmark.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Self-Hostingwith Ollama**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-13_5.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-13.5.html` - regenerate this notebook whenever the source HTML is updated.*
